# SQL Refresh: Exploring the IMDb Database

**Course:** Database Applications Development  
**Medina County Career Center**

---

Welcome back to SQL! We've been away from databases for a bit, so today we're going to dust off our SQL skills with a brand new database — **IMDb movie data**.

This database has ~19,000 movies and TV series from 2010 to present, along with ratings, actors, directors, and crew. It's all titles you'll actually recognize.

**Today's goals:**
1. Reconnect to a database and remember how it works
2. Explore the structure of a new database
3. Practice SELECT, WHERE, ORDER BY, and GROUP BY
4. Find some movies and actors YOU care about

## Setup

First, let's import our libraries and connect to the database. Same pattern as before — nothing new here.

In [ ]:
# -------------------------------------------------------
# WHAT THIS DOES:
# Imports two libraries we need:
#   - pandas (pd): lets us display query results as nice tables
#   - sqlite3: lets Python talk to a SQLite database file
#
# Then we open (connect to) the database file and create a
# 'cursor' — think of it as a pointer we use to send SQL
# commands. Finally, we print a message to confirm it worked.
# -------------------------------------------------------

import pandas as pd
import sqlite3

# Connect to the filtered IMDb database
dbPath = 'imdb_class_2010plus.db'
connection = sqlite3.connect(dbPath)
cursor = connection.cursor()

print(f'Connected to: {dbPath}')

---

## Part 1: What's in This Database?

Before we write any queries, let's see what tables we have. Remember `sqlite_master`? That's the system table that tells us what's inside the database.

In [ ]:
# -------------------------------------------------------
# WHAT THIS DOES:
# Asks the database to list every table it contains.
# sqlite_master is a special built-in table that stores
# information about the database structure itself.
#
# After getting the list of table names, we loop through
# each one and run a COUNT(*) query to find out how many
# rows of data are stored in each table.
#
# SQL used: SELECT name FROM sqlite_master WHERE type='table'
# -------------------------------------------------------

# List all tables in the database
tableQuery = "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;"
tables = cursor.execute(tableQuery).fetchall()

print("Tables in imdb_class_2010plus.db:")
print("-" * 35)
for i, (tableName,) in enumerate(tables, 1):
    # Also get the row count for each table
    count = cursor.execute(f"SELECT COUNT(*) FROM {tableName}").fetchone()[0]
    print(f"  {i}. {tableName:25s} ({count:,} rows)")

### Quick Overview

Here's what each table holds:

| Table | What it stores | Think of it as... |
|-------|---------------|-------------------|
| **title_basics** | Movie/show info (title, year, genre, runtime) | The Netflix browse page |
| **title_ratings** | IMDb ratings and vote counts | The star ratings you see on IMDb |
| **title_principals** | Who worked on what (actors, directors, etc.) | The movie credits |
| **title_crew_person** | Directors and writers for each title | The "Directed by" credit |
| **name_basics** | People's names, birth years, professions | The actor/director profiles |

Let's look at each one.

### Table 1: title_basics

This is the main table — one row per movie or TV series.

In [ ]:
# -------------------------------------------------------
# WHAT THIS DOES:
# PRAGMA table_info() is a special SQLite command that
# returns the column names and data types for a given table.
# This is how you "peek inside" a table to see its structure
# before writing queries against it.
#
# Then SELECT * FROM title_basics LIMIT 5 pulls the first
# 5 rows so we can see what the actual data looks like.
#
# SQL used: PRAGMA table_info(table_name)
#           SELECT * FROM table_name LIMIT 5
# -------------------------------------------------------

# Look at the columns and a few sample rows
print("=== title_basics ===\n")

# Show column info
schema = cursor.execute("PRAGMA table_info(title_basics)").fetchall()
for col in schema:
    print(f"  {col[1]:20s} {col[2]}")

print()

# Sample rows
df = pd.read_sql_query("SELECT * FROM title_basics LIMIT 5", connection)
print(df.to_string(index=False))

**Key columns to know:**
- `tconst` — the unique ID for each title (like "tt1375666" for Inception). Think of this like a student ID but for movies.
- `titleType` — either "movie" or "tvSeries"
- `primaryTitle` — the title you'd recognize
- `startYear` — when it came out
- `genres` — comma-separated list like "Action,Adventure,Sci-Fi"
- `runtimeMinutes` — how long it is

### Table 2: title_ratings

Every title's IMDb rating and how many people voted.

In [ ]:
# -------------------------------------------------------
# WHAT THIS DOES:
# Same pattern as above — uses PRAGMA to inspect the columns
# of title_ratings, then pulls 5 sample rows.
#
# Notice that title_ratings also has a tconst column.
# This is the foreign key (FK) that links each rating row
# back to its matching movie in title_basics.
#
# SQL used: PRAGMA table_info(title_ratings)
#           SELECT * FROM title_ratings LIMIT 5
# -------------------------------------------------------

# Sample from title_ratings
print("=== title_ratings ===\n")

schema = cursor.execute("PRAGMA table_info(title_ratings)").fetchall()
for col in schema:
    print(f"  {col[1]:20s} {col[2]}")

print()

df = pd.read_sql_query("SELECT * FROM title_ratings LIMIT 5", connection)
print(df.to_string(index=False))

**Key columns:**
- `tconst` — matches up with title_basics (same movie ID)
- `averageRating` — the IMDb score (1.0 to 10.0)
- `numVotes` — how many people rated it (more votes = more trustworthy rating)

### Table 3: title_principals

This connects **people** to **titles**. If Nicolas Cage is in a movie, there's a row here linking his ID to that movie's ID.

In [ ]:
# -------------------------------------------------------
# WHAT THIS DOES:
# Inspects title_principals — the bridge table that connects
# people (name_basics) to movies/shows (title_basics).
#
# This table has TWO foreign keys:
#   tconst -> points to a title in title_basics
#   nconst -> points to a person in name_basics
#
# The 'ordering' column is just a rank (1 = top billed actor).
# The 'characters' column stores the character name as JSON.
#
# SQL used: PRAGMA table_info(title_principals)
#           SELECT * FROM title_principals LIMIT 5
# -------------------------------------------------------

# Sample from title_principals
print("=== title_principals ===\n")

schema = cursor.execute("PRAGMA table_info(title_principals)").fetchall()
for col in schema:
    print(f"  {col[1]:20s} {col[2]}")

print()

df = pd.read_sql_query("SELECT * FROM title_principals LIMIT 5", connection)
print(df.to_string(index=False))

**Key columns:**
- `tconst` — the movie/show ID (links to title_basics)
- `nconst` — the person ID (links to name_basics)
- `category` — what they did: "actor", "actress", "director", "writer", "producer", etc.
- `characters` — the character name(s) they played

This is the table that lets us answer questions like "What movies was Samuel L. Jackson in?" or "Who acted in Inception?"

### Table 4: name_basics

The people table — actors, directors, writers, etc.

In [ ]:
# -------------------------------------------------------
# WHAT THIS DOES:
# Inspects name_basics — the table that stores information
# about every person in the database (actors, directors,
# writers, producers, etc.).
#
# The primary key here is nconst — a unique person ID
# assigned by IMDb (e.g., nm0000115 = Nicolas Cage).
#
# knownForTitles stores comma-separated tconst values,
# listing the titles each person is most famous for.
#
# SQL used: PRAGMA table_info(name_basics)
#           SELECT * FROM name_basics LIMIT 5
# -------------------------------------------------------

# Sample from name_basics
print("=== name_basics ===\n")

schema = cursor.execute("PRAGMA table_info(name_basics)").fetchall()
for col in schema:
    print(f"  {col[1]:20s} {col[2]}")

print()

df = pd.read_sql_query("SELECT * FROM name_basics LIMIT 5", connection)
print(df.to_string(index=False))

**Key columns:**
- `nconst` — unique person ID (like "nm0000115" for Nicolas Cage)
- `primaryName` — their name
- `birthYear` / `deathYear` — when they were born (and died, if applicable)
- `primaryProfession` — what they're known for
- `knownForTitles` — comma-separated list of their most famous title IDs

### Table 5: title_crew_person

Directors and writers for each title. Similar to title_principals but specifically for crew roles.

In [ ]:
# -------------------------------------------------------
# WHAT THIS DOES:
# Inspects title_crew_person — a focused bridge table that
# links directors and writers to their titles.
#
# Like title_principals, it has two foreign keys:
#   tconst -> title_basics (which movie/show)
#   nconst -> name_basics  (which person)
#
# The 'role' column is either 'director' or 'writer'.
# This table is more specific than title_principals —
# use it when you only care about who directed or wrote
# a film.
#
# SQL used: PRAGMA table_info(title_crew_person)
#           SELECT * FROM title_crew_person LIMIT 5
# -------------------------------------------------------

# Sample from title_crew_person
print("=== title_crew_person ===\n")

schema = cursor.execute("PRAGMA table_info(title_crew_person)").fetchall()
for col in schema:
    print(f"  {col[1]:20s} {col[2]}")

print()

df = pd.read_sql_query("SELECT * FROM title_crew_person LIMIT 5", connection)
print(df.to_string(index=False))

---

## Part 2: SQL Refresher — The Basics

Let's warm up with queries you've seen before. Same SQL, new data.

### SELECT + WHERE

Remember: `SELECT` picks columns, `WHERE` filters rows.

Let's find some movies you know.

In [ ]:
# -------------------------------------------------------
# WHAT THIS DOES:
# Uses SELECT to pick four specific columns from title_basics,
# then WHERE to filter down to only the row where the title
# is exactly 'Inception'.
#
# pd.read_sql_query() runs the SQL and returns the result
# as a pandas DataFrame so it prints cleanly as a table.
#
# SQL used: SELECT col1, col2 FROM table WHERE col = 'value'
# -------------------------------------------------------

# Find Inception
query = '''
    SELECT primaryTitle, startYear, genres, runtimeMinutes
    FROM title_basics
    WHERE primaryTitle = 'Inception'
'''

df = pd.read_sql_query(query, connection)
print(df.to_string(index=False))

#### New SQL Concept: `LIKE` with Wildcards

Sometimes you don't know the exact title — you just want to search for a keyword. That's what `LIKE` does.

- `LIKE '%Avengers%'` — finds any title that **contains** "Avengers" anywhere
- `LIKE 'The%'` — finds titles that **start with** "The"
- `LIKE '%Man'` — finds titles that **end with** "Man"

The `%` is a **wildcard** — it means "any number of characters." Think of it like the `*` in a Google search.

**Important:** `LIKE` is **case-sensitive** in SQLite by default for ASCII characters. So `'spider'` won't match `'Spider'`.

In [ ]:
# -------------------------------------------------------
# WHAT THIS DOES:
# Uses LIKE with the % wildcard to search for any title
# that contains the word 'Avengers' anywhere in it.
#
# '%Avengers%' means:
#   - anything before 'Avengers' (the first %)
#   - the word 'Avengers'
#   - anything after 'Avengers' (the second %)
#
# ORDER BY startYear sorts the results from oldest to newest.
#
# SQL used: WHERE column LIKE '%keyword%'
#           ORDER BY column
# -------------------------------------------------------

# Find all Marvel movies (titles containing "Avengers")
query = '''
    SELECT primaryTitle, startYear, genres
    FROM title_basics
    WHERE primaryTitle LIKE '%Avengers%'
    ORDER BY startYear
'''

df = pd.read_sql_query(query, connection)
print(df.to_string(index=False))

### Try It: Find Your Favorites

Change the query below to search for a movie or show YOU like. Try:
- An exact title: `WHERE primaryTitle = 'Your Movie'`
- A partial match: `WHERE primaryTitle LIKE '%keyword%'`

In [ ]:
# -------------------------------------------------------
# WHAT THIS DOES:
# Same LIKE wildcard search, but this time searching for
# 'Spider' — which catches titles like Spider-Man,
# Spider-Verse, Spiderhead, etc.
#
# This is a student practice cell — change 'Spider' to any
# keyword that matches a movie or show you're interested in.
#
# SQL used: WHERE column LIKE '%keyword%'
#           ORDER BY column
# -------------------------------------------------------

# YOUR TURN: Search for a movie or show you like
query = '''
    SELECT primaryTitle, startYear, genres, runtimeMinutes
    FROM title_basics
    WHERE primaryTitle LIKE '%Spider%'
    ORDER BY startYear
'''

df = pd.read_sql_query(query, connection)
print(df.to_string(index=False))

### ORDER BY + LIMIT

Remember: `ORDER BY` sorts results, `LIMIT` caps how many rows you get back.

#### New SQL Concepts: `JOIN`, `ON`, and Table Aliases

The next query pulls data from **two tables at once**. To do that, we need a few new tools:

**`JOIN ... ON`** — Combines rows from two tables where a column matches.
```sql
FROM title_basics b
JOIN title_ratings r ON b.tconst = r.tconst
```
This says: "Take each row from `title_basics`, find the matching row in `title_ratings` where the `tconst` values are the same, and glue them together."

**Table Aliases** — The `b` and `r` are shortcuts (aliases) so we don't have to type the full table name every time:
- `b` = `title_basics`
- `r` = `title_ratings`

Then we use `b.primaryTitle` instead of `title_basics.primaryTitle`. Same thing, less typing.

**Why do we need aliases?** Both tables have a column called `tconst`. If we just write `tconst`, SQL doesn't know which table we mean. The alias makes it clear: `b.tconst` vs `r.tconst`.

In [ ]:
# -------------------------------------------------------
# WHAT THIS DOES:
# Joins two tables — title_basics (movie info) and
# title_ratings (scores) — using the shared tconst column.
#
# The JOIN ON line is the glue: match every movie in
# title_basics to its rating row in title_ratings.
#
# WHERE filters to only movies (not TV series) with
# more than 100,000 votes, so obscure titles don't
# sneak into our top 10.
#
# ORDER BY averageRating DESC sorts highest-rated first.
# LIMIT 10 keeps only the top 10 results.
#
# SQL used: JOIN table ON table1.col = table2.col
#           WHERE col = value AND col > value
#           ORDER BY col DESC
#           LIMIT n
# -------------------------------------------------------

# Top 10 highest-rated movies with at least 100,000 votes
# (We join title_basics and title_ratings to get both title info AND ratings)
query = '''
    SELECT b.primaryTitle, b.startYear, r.averageRating, r.numVotes
    FROM title_basics b
    JOIN title_ratings r ON b.tconst = r.tconst
    WHERE b.titleType = 'movie'
      AND r.numVotes > 100000
    ORDER BY r.averageRating DESC
    LIMIT 10
'''

df = pd.read_sql_query(query, connection)
print("Top 10 highest-rated movies (100K+ votes):")
print(df.to_string(index=False))

#### New SQL Concept: `IS NOT NULL`

Some movies are missing their runtime — the value is `NULL` (blank/unknown). If we're sorting by runtime, we want to skip those.

```sql
WHERE runtimeMinutes IS NOT NULL
```

This filters out any row where `runtimeMinutes` has no value. You can't use `= NULL` or `!= NULL` in SQL — it **must** be `IS NULL` or `IS NOT NULL`. That's just how SQL works.

In [ ]:
# -------------------------------------------------------
# WHAT THIS DOES:
# Finds the longest movies in the database by sorting
# runtimeMinutes from largest to smallest (DESC).
#
# IS NOT NULL is required here — some movies don't have
# a runtime entered. Without this filter, NULL values
# would appear at the top of a DESC sort (SQLite treats
# NULL as very large). IS NOT NULL skips those blank rows.
#
# LIMIT 10 gives us the top 10 longest films.
#
# SQL used: WHERE column IS NOT NULL
#           ORDER BY column DESC
#           LIMIT n
# -------------------------------------------------------

# Longest movies in the database
query = '''
    SELECT primaryTitle, startYear, runtimeMinutes, genres
    FROM title_basics
    WHERE runtimeMinutes IS NOT NULL
      AND titleType = 'movie'
    ORDER BY runtimeMinutes DESC
    LIMIT 10
'''

df = pd.read_sql_query(query, connection)
print("Longest movies:")
print(df.to_string(index=False))

### Try It: Your Own Ranking

Modify the query below to find YOUR top 10. Some ideas:
- Most popular movies from a specific year
- Highest rated TV series
- Shortest movies
- Movies from a specific genre (hint: `WHERE genres LIKE '%Horror%'`)

### Best genres to try first
- Action
- Comedy
- Drama
- Horror
- Thriller
- Sci-Fi
- Adventure
- Animation
- Crime

### Best genre combinations to try
- Action,Adventure
- Action,Comedy
- Action,Sci-Fi
- Action,Thriller
- Adventure,Fantasy
- Adventure,Sci-Fi
- Comedy,Drama
- Comedy,Romance
- Crime,Drama
- Horror,Thriller
- Mystery,Thriller

In [ ]:
# -------------------------------------------------------
# WHAT THIS DOES:
# Joins title_basics and title_ratings, then filters to
# movies in the 'Adventure' genre with at least 10,000
# votes. Results are sorted by average rating (highest
# first) and cut off at 10 rows.
#
# genres LIKE '%Adventure%' works because genres is stored
# as a comma-separated string like 'Action,Adventure,Sci-Fi'.
# The % wildcard catches Adventure anywhere in that string.
#
# STUDENT CHALLENGE: Change 'Adventure' to any genre from
# the list above and rerun to build your own custom top 10.
#
# SQL used: JOIN, WHERE with LIKE and AND, ORDER BY DESC, LIMIT
# -------------------------------------------------------

# YOUR TURN: Create your own top 10 list
query = '''
    SELECT b.primaryTitle, b.startYear, b.genres, r.averageRating, r.numVotes
    FROM title_basics b
    JOIN title_ratings r ON b.tconst = r.tconst
    WHERE b.titleType = 'movie'
      AND b.genres LIKE '%Adventure%'
      AND r.numVotes > 10000
    ORDER BY r.averageRating DESC
    LIMIT 10
'''

df = pd.read_sql_query(query, connection)
print(df.to_string(index=False))

---

### GROUP BY + Aggregations

Remember: `GROUP BY` groups rows together so you can use `COUNT()`, `AVG()`, `MAX()`, `MIN()`, `SUM()` on each group.

#### New SQL Concepts: `ROUND()` and `AVG()`

You already know `COUNT(*)` counts rows. Two more aggregation functions:

**`AVG(column)`** — Calculates the average (mean) of a numeric column.
```sql
AVG(r.averageRating)   -- average of all ratings in the group
```

**`ROUND(value, decimals)`** — Rounds a number to a specified number of decimal places.
```sql
ROUND(AVG(r.averageRating), 2)   -- average rounded to 2 decimal places
ROUND(AVG(r.numVotes), 0)        -- average rounded to 0 decimals (whole number)
```

Without `ROUND()`, you'd get numbers like `5.873291046...` which are hard to read. `ROUND(..., 2)` gives you `5.87`.

In [ ]:
# -------------------------------------------------------
# WHAT THIS DOES:
# Uses GROUP BY to group all movies by the year they came
# out (startYear), then COUNT(*) counts how many movies
# are in each year group.
#
# WHERE startYear >= 2015 limits results to recent years
# so the output isn't too long.
#
# ORDER BY startYear sorts the results chronologically.
#
# SQL used: GROUP BY column
#           COUNT(*) AS alias
#           WHERE column >= value
# -------------------------------------------------------

# How many movies came out each year?
query = '''
    SELECT startYear, COUNT(*) as movieCount
    FROM title_basics
    WHERE titleType = 'movie'
      AND startYear >= 2015
    GROUP BY startYear
    ORDER BY startYear
'''

df = pd.read_sql_query(query, connection)
print("Movies per year (2015+):")
print(df.to_string(index=False))

In [ ]:
# -------------------------------------------------------
# WHAT THIS DOES:
# Builds on the previous query by also joining title_ratings
# so we can calculate the average IMDb rating and average
# vote count per year.
#
# AVG(r.averageRating) calculates the mean rating across
# all movies in that year group.
#
# ROUND(..., 2) and ROUND(..., 0) clean up the decimal
# places so the numbers are easy to read.
#
# This lets us answer: are older movies in the database
# rated higher or lower than recent ones?
#
# SQL used: GROUP BY with multiple aggregates
#           AVG(), ROUND(), COUNT() together
# -------------------------------------------------------

# Average rating by year — are movies getting better or worse?
query = '''
    SELECT b.startYear, 
           COUNT(*) as movieCount,
           ROUND(AVG(r.averageRating), 2) as avgRating,
           ROUND(AVG(r.numVotes), 0) as avgVotes
    FROM title_basics b
    JOIN title_ratings r ON b.tconst = r.tconst
    WHERE b.titleType = 'movie'
      AND b.startYear >= 2015
    GROUP BY b.startYear
    ORDER BY b.startYear
'''

df = pd.read_sql_query(query, connection)
print("Movie ratings by year:")
print(df.to_string(index=False))

In [ ]:
# -------------------------------------------------------
# WHAT THIS DOES:
# Groups all titles by titleType (either 'movie' or
# 'tvSeries') and calculates two things for each group:
#   1. COUNT(*) — total number of titles of that type
#   2. AVG(runtimeMinutes) — average length in minutes
#
# WHERE runtimeMinutes IS NOT NULL filters out rows with
# missing runtime data so they don't skew the average.
#
# STUDENT CHALLENGE: Try adding AVG(r.averageRating) by
# joining title_ratings — do movies or TV shows rate higher?
#
# SQL used: GROUP BY, COUNT(*), ROUND(AVG(...), 1),
#           WHERE column IS NOT NULL
# -------------------------------------------------------

# YOUR TURN: Write a GROUP BY query
query = '''
    SELECT titleType, 
           COUNT(*) as count,
           ROUND(AVG(runtimeMinutes), 1) as avgRuntime
    FROM title_basics
    WHERE runtimeMinutes IS NOT NULL
    GROUP BY titleType
'''

df = pd.read_sql_query(query, connection)
print(df.to_string(index=False))

---

## Part 3: Connecting Tables (JOINs)

The real power of this database comes when we connect tables together. We've been doing this already with `JOIN` — let's make it explicit.

#### New SQL Concepts: `IN ()` and `DISTINCT`

Two more tools you'll see in the JOIN queries below:

**`IN (value1, value2, ...)`** — A shortcut for multiple `OR` conditions.
```sql
WHERE p.category IN ('actor', 'actress')
```
This is the same as writing:
```sql
WHERE p.category = 'actor' OR p.category = 'actress'
```
Much cleaner when you have several values to check!

**`DISTINCT`** — Removes duplicate values from a result.
```sql
COUNT(DISTINCT p.tconst)   -- counts unique movie IDs only
```
Without `DISTINCT`, if an actor has two credits on the same movie (like appearing in two roles), it would count that movie twice. `DISTINCT` makes sure each movie is counted only once.

In [ ]:
# -------------------------------------------------------
# WHAT THIS DOES:
# Joins name_basics (people) to title_principals (credits)
# to count how many unique titles each actor/actress
# appears in.
#
# IN ('actor', 'actress') filters to acting credits only
# (skips directors, writers, producers, etc.).
#
# COUNT(DISTINCT p.tconst) counts each movie only once,
# even if an actor had multiple roles in the same film.
#
# GROUP BY n.nconst groups by person ID (not name, in case
# two people share the same name).
#
# ORDER BY movieCount DESC puts the most prolific actors first.
#
# SQL used: JOIN, WHERE IN (...),
#           GROUP BY, COUNT(DISTINCT col), ORDER BY DESC
# -------------------------------------------------------

# Who are the most prolific actors in this database?
query = '''
    SELECT n.primaryName, COUNT(DISTINCT p.tconst) as movieCount
    FROM name_basics n
    JOIN title_principals p ON n.nconst = p.nconst
    WHERE p.category IN ('actor', 'actress')
    GROUP BY n.nconst
    ORDER BY movieCount DESC
    LIMIT 15
'''

df = pd.read_sql_query(query, connection)
print("Most prolific actors (by number of titles):")
print(df.to_string(index=False))

In [ ]:
# -------------------------------------------------------
# WHAT THIS DOES:
# A 3-table JOIN that finds all movies a specific actor
# appeared in, along with their ratings.
#
# Join chain:
#   name_basics (n) -> title_principals (p) -> title_basics (b)
#                                           -> title_ratings (r)
#
# WHERE n.primaryName = 'Nicolas Cage' finds that one person.
# AND p.category = 'actor' ensures we only count acting roles.
# AND b.titleType = 'movie' skips TV appearances.
#
# ORDER BY b.startYear DESC shows newest movies first.
#
# SQL used: 3-table JOIN, WHERE with multiple AND conditions,
#           ORDER BY DESC
# -------------------------------------------------------

# What movies has a specific actor been in?
# Let's try Nicolas Cage
query = '''
    SELECT b.primaryTitle, b.startYear, b.genres, r.averageRating
    FROM name_basics n
    JOIN title_principals p ON n.nconst = p.nconst
    JOIN title_basics b ON p.tconst = b.tconst
    JOIN title_ratings r ON b.tconst = r.tconst
    WHERE n.primaryName = 'Nicolas Cage'
      AND p.category = 'actor'
      AND b.titleType = 'movie'
    ORDER BY b.startYear DESC
'''

df = pd.read_sql_query(query, connection)
print(f"Nicolas Cage movies ({len(df)} total):")
print(df.to_string(index=False))

### Try It: Look Up YOUR Favorite Actor

Change the name in the query below to find movies for an actor you like.

In [ ]:
# -------------------------------------------------------
# WHAT THIS DOES:
# Same 3-table JOIN as above, but this time we're searching
# for Zendaya and using IN ('actor', 'actress') to catch
# both credit categories she might appear under.
#
# The results include the characters column from
# title_principals, which shows what character she played.
#
# STUDENT CHALLENGE: Replace 'Zendaya' with any actor or
# actress name — use the exact name as it appears on IMDb.
#
# SQL used: 3-table JOIN, WHERE IN (...), ORDER BY DESC
# -------------------------------------------------------

# YOUR TURN: Pick an actor and find their movies
query = '''
    SELECT b.primaryTitle, b.startYear, r.averageRating, p.characters
    FROM name_basics n
    JOIN title_principals p ON n.nconst = p.nconst
    JOIN title_basics b ON p.tconst = b.tconst
    JOIN title_ratings r ON b.tconst = r.tconst
    WHERE n.primaryName = 'Zendaya'
      AND p.category IN ('actor', 'actress')
    ORDER BY b.startYear DESC
'''

df = pd.read_sql_query(query, connection)
print(df.to_string(index=False))

In [ ]:
# -------------------------------------------------------
# WHAT THIS DOES:
# Flips the JOIN direction — instead of starting from a
# person and finding their movies, we start from a specific
# movie title and find everyone who worked on it.
#
# Join chain:
#   title_basics (b) -> title_principals (p) -> name_basics (n)
#
# WHERE b.primaryTitle = 'Inception' pins us to that movie.
# IN ('actor', 'actress', 'director') filters to key credits.
# ORDER BY p.ordering sorts by billing order (star = 1).
#
# SQL used: 3-table JOIN starting from title_basics,
#           WHERE IN (...), ORDER BY ordering
# -------------------------------------------------------

# Who starred in a specific movie?
query = '''
    SELECT n.primaryName, p.category, p.characters
    FROM title_basics b
    JOIN title_principals p ON b.tconst = p.tconst
    JOIN name_basics n ON p.nconst = n.nconst
    WHERE b.primaryTitle = 'Inception'
      AND p.category IN ('actor', 'actress', 'director')
    ORDER BY p.ordering
'''

df = pd.read_sql_query(query, connection)
print("Cast & director of Inception:")
print(df.to_string(index=False))

### Try It: Who's in YOUR Favorite Movie?

Change the title below to look up the cast of a movie you love.

In [ ]:
# -------------------------------------------------------
# WHAT THIS DOES:
# Same cast-lookup query as above, but pointed at
# 'The Batman' instead of 'Inception'.
#
# This is a student practice cell. Change 'The Batman'
# to any movie title in the database. Use the exact title
# as it appears on IMDb (capitalization matters).
#
# STUDENT CHALLENGE: Try a movie YOU love. If the title
# has special characters or a colon, you may need to
# adjust — try LIKE '%keyword%' if exact match returns nothing.
#
# SQL used: 3-table JOIN, WHERE with exact title match,
#           IN (...), ORDER BY p.ordering
# -------------------------------------------------------

# YOUR TURN: Look up the cast of a movie
query = '''
    SELECT n.primaryName, p.category, p.characters
    FROM title_basics b
    JOIN title_principals p ON b.tconst = p.tconst
    JOIN name_basics n ON p.nconst = n.nconst
    WHERE b.primaryTitle = 'The Batman'
      AND p.category IN ('actor', 'actress', 'director')
    ORDER BY p.ordering
'''

df = pd.read_sql_query(query, connection)
print(df.to_string(index=False))

---

## Wrap-Up

**Today we refreshed:**
- `SELECT` / `WHERE` / `ORDER BY` / `LIMIT` — picking and filtering data
- `GROUP BY` with `COUNT()`, `AVG()` — summarizing data
- `JOIN` — connecting tables together

**The IMDb database has 5 tables:**
- `title_basics` and `title_ratings` hold movie/show info
- `name_basics` holds people info
- `title_principals` and `title_crew_person` connect people to titles

**Next up:** Your task notebook — explore this database on your own!

In [ ]:
# -------------------------------------------------------
# WHAT THIS DOES:
# Closes the connection to the database file.
# Always do this at the end of your notebook — it's like
# logging out of a system. Leaving connections open can
# cause issues if another process tries to access the file.
# -------------------------------------------------------

# Clean up
connection.close()
print("Database connection closed.")